# geo1261 – Mission plan for the Green Fab Lab

**Unit:** geo1260 Drone mapping · **Slides:** `geo1260_Drone_Imagery_WebODM` (part *Plan*) · **Data:** `data/geo1261/gfl_flight_area.gpkg` · **Tools:** `geo1261_Mission_Tools.py` · **Homework**

Before a mapping flight the planning app computes how high to fly, how far apart the flight lines are, where the camera takes a photo, how fast the drone may fly and how long the flight takes. In this notebook you compute the key numbers yourself for the Mavic 3M over the Green Fab Lab, estimate the plan by hand, let the given tools draw the flight lines, and export the area for the flight app (DJI Pilot 2). In the field you compare your numbers with the app.

You will learn to
1. derive the ground sampling distance (GSD) and the photo footprint from the camera specs,
2. turn height and overlap into line spacing, photo spacing, speed and flight time,
3. explain why the multispectral camera decides the plan when all cameras record,
4. choose a flight height and justify it.

You program sections 1–3 and the table in section 6. The geometry (flight lines over the polygon) is given in `geo1261_Mission_Tools.py` (imported as `mt`); from section 3 on the notebook uses those reference functions, so a mistake in an early exercise does not block the rest. Writing the flight lines yourself is the stretch task in section 7.

About 2 h, plus 1 h for the stretch task.

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import geo1261_Mission_Tools as mt         # given tools (same folder as this notebook)

DATA = Path("data") / "geo1261"
area_gdf = gpd.read_file(DATA / "gfl_flight_area.gpkg", layer="flight_area")
bld_gdf = gpd.read_file(DATA / "gfl_flight_area.gpkg", layer="building")
area = area_gdf.geometry.iloc[0]           # shapely Polygon in EPSG:25832 (metres)
print(area_gdf.crs, f"{area.area:.0f} m²")

ax = area_gdf.plot(color="pink", edgecolor="crimson")
bld_gdf.plot(ax=ax, color="grey")
ax.set_title("Green Fab Lab: flight area and building");

## 1. Camera: pixel size on the ground

DJI Mavic 3M specs (ag.dji.com/mavic-3-m/specs):

| | RGB | multispectral (each band) |
|---|---|---|
| image size | 5280 × 3956 px | 2592 × 1944 px |
| field of view | 84° **diagonal** | 61.2° × 48.1° (horizontal × vertical) |
| shortest photo interval | 0.7 s (JPEG) | 2 s (TIFF) |

A camera with field of view $\alpha$ at height $H$ sees a ground width of $2H\tan(\alpha/2)$. Divided by the number of pixels across that width you get the **ground sampling distance** (GSD).

**Exercise 1:** complete `CAMERAS` with `gsd_per_m` (GSD in metres per metre of height) for both cameras, and write `gsd(cam, height)`. Mind which field of view goes with which pixel count.

In [ ]:
CAMERAS = {
    "RGB": dict(px=(5280, 3956), interval=0.7),
    "MS": dict(px=(2592, 1944), interval=2.0),
}
# YOUR CODE HERE

for h in (25, 40, 60, 120):
    print(f"{h:4d} m: RGB {gsd('RGB', h) * 100:.2f} cm, MS {gsd('MS', h) * 100:.2f} cm")

mt.check_gsd(gsd)

**Check:** at 100 m the RGB pixel should be about 2.7 cm. Why is the multispectral pixel larger?

✍️ *Your answer:*

## 2. From height and overlap to footprint, spacing and speed

The GSD is the ground of **one pixel**; the **footprint** is the ground of the **whole photo**:

$$\text{footprint} = \text{GSD} \times \text{number of pixels}$$

for each image side. We assume the long image side lies **across** the flight direction (as in DJI's mapping missions).

* footprint across = pixels across × GSD, footprint along = pixels along × GSD
* line spacing = footprint across × (1 − side overlap)
* photo spacing ("base") = footprint along × (1 − front overlap)
* speed = photo spacing / shortest photo interval, at most `V_MAX`

**Exercise 2:** write `plan_numbers(cam, height, front, side)` that returns a dict with `gsd`, `across`, `along`, `spacing`, `base` and `v`.

In [ ]:
V_MAX = 15.0    # m/s

# YOUR CODE HERE

H = 25
pd.DataFrame({cam: plan_numbers(cam, H) for cam in CAMERAS}).round(3)

mt.check_plan_numbers(plan_numbers)

**Questions:** Both cameras record at the same time. Which camera sets the line spacing, which the photo spacing and which the speed? What do you use for the flight?

✍️ *Your answer:*

## 3. Estimate the plan first

From here on we use the reference functions in `mt` (same formulas as yours, if your checks said OK).

The area is not a rectangle, but its **minimum rotated rectangle** is a good first guess. The drone flies along its long side (fewer turns). The next cell gives you the rectangle and the numbers for 25 m, all cameras.

**Exercise 3:** estimate with plain arithmetic (no Shapely):
* `n_lines`: how many lines `spacing` apart cover the width? (round up)
* `photos_per_line`: one photo at the start, then one every `base` metres along the length (round down, + 1)
* `n_photos`, and `est_length`: all lines + the connections between them (each about one `spacing`)
* `est_minutes`: `est_length / v` plus `mt.TURN_S` seconds for each turn, in minutes

Write your numbers down **before** you run section 4.

In [ ]:
H = 25
L, W = mt.rectangle_size(area)
heading = mt.heading_of(area)
c = mt.combined(H)                          # all cameras: the smallest spacing, base and speed win
spacing, base, v = c["spacing"], c["base"], c["v"]
print(f"rectangle {L:.0f} m long × {W:.0f} m wide, heading {heading:.0f}° from east")
print(f"{H} m, all cameras: spacing {spacing:.1f} m, base {base:.1f} m, speed {v:.1f} m/s")

In [ ]:
# YOUR CODE HERE
print(f"estimate: {n_lines} lines, {n_photos} photos, {est_length:.0f} m, about {est_minutes:.1f} min")

## 4. The plan over the real area

`mt.flight_lines` rotates the area so that the flight direction becomes the x axis, draws lines `spacing` apart, cuts them with the area and rotates them back (every second line reversed: the drone flies back and forth). `mt.photo_positions` puts a photo every `base` metres along each line, `mt.path_length` adds the lines and the connections.

Run the cell and compare with your estimate.

In [ ]:
lines = mt.flight_lines(area, spacing, heading)
photos = mt.photo_positions(lines, base)
length = mt.path_length(lines)
minutes = mt.flight_minutes(length, v, len(lines))

pd.DataFrame({"estimate (rectangle)": [n_lines, n_photos, est_length, est_minutes],
              "plan (polygon)": [len(lines), len(photos), length, minutes]},
             index=["lines", "photos", "path m", "minutes"]).round(1)

**What the clipping does:** the orange dashed lines are your estimate (full-length lines over the rectangle), the blue lines the plan (the same lines cut with the area polygon by `LineString.intersection`). Every metre of orange line outside the pink area is flight time and photos the plan saves.

In [ ]:
rect = area.minimum_rotated_rectangle
rect_lines = mt.flight_lines(rect, spacing, heading)          # the lines of your estimate

ax = area_gdf.plot(color="pink", edgecolor="crimson", figsize=(7, 7))
gpd.GeoSeries([rect], crs=25832).boundary.plot(ax=ax, color="grey", linestyle=":")
gpd.GeoSeries(rect_lines, crs=25832).plot(ax=ax, color="darkorange", linestyle="--", linewidth=3)
gpd.GeoSeries(lines, crs=25832).plot(ax=ax, color="navy", linewidth=2.5)
ax.set_title(f"Estimate (rectangle, orange dashed) vs plan (polygon, blue)\n"
             f"line length {sum(l.length for l in rect_lines):.0f} m vs {sum(l.length for l in lines):.0f} m");

**Question:** Where does the plan differ from your estimate, and why? Which number would you trust for the battery?

✍️ *Your answer:*

## 5. Map and export

The map shows the area, the flight lines, the photo positions and the footprints of one RGB and one multispectral photo. Then we save the plan as a GeoPackage (for QGIS) and the **area** as KML for DJI Pilot 2 (the app plans its own lines inside it).

In [ ]:
ax = area_gdf.plot(color="pink", edgecolor="crimson", figsize=(7, 7))
bld_gdf.plot(ax=ax, color="grey")
gpd.GeoSeries(lines, crs=25832).plot(ax=ax, color="navy")
gpd.GeoSeries(photos, crs=25832).plot(ax=ax, color="navy", markersize=6)
p = photos[len(photos) // 2]
gpd.GeoSeries([mt.footprint(p, "RGB", H, heading)]).boundary.plot(ax=ax, color="grey")
gpd.GeoSeries([mt.footprint(p, "MS", H, heading)]).boundary.plot(ax=ax, color="tab:blue", linestyle="--")
ax.set_title(f"Mission plan, {H} m, all cameras");

out = Path("geo1261_plan.gpkg")
gpd.GeoDataFrame({"line": range(1, len(lines) + 1)}, geometry=lines, crs=25832).to_file(out, layer="flight_lines")
gpd.GeoDataFrame({"photo": range(1, len(photos) + 1)}, geometry=photos, crs=25832).to_file(out, layer="photos")
area_gdf.to_crs(4326).to_file("gfl_flight_area.kml", driver="KML")   # for DJI Pilot 2 (WGS 84)
print("written", out, "and gfl_flight_area.kml")

## 6. Compare heights

`mt.mission(area, height, cams)` does sections 2–4 in one call and returns one table row (a dict).

**Exercise 4:** make a table for 25, 30, 40 and 60 m, once for all cameras `("RGB", "MS")` and once for RGB only `("RGB",)`: call `mt.mission` for every combination, collect the rows in a list and turn it into a `DataFrame` called `table`.

In [ ]:
# YOUR CODE HERE
table.round(2)

**Questions**
1. Which height would you choose for the Green Fab Lab, and why? Think of pixel size, the size of the area (how much of a photo lies outside it?), the trees and the battery.
2. At 25 m a photo covers more than a third of the area. What does that mean for the photos at the edge of the area, and for the GCPs near the edge?
3. Stretch: change the overlap to 83 % / 75 % (vegetation, ODM). How much longer is the flight?
4. In the field: write down DJI Pilot 2's numbers for the same height and overlap (line spacing, number of photos, time). Where do they differ from yours, and why?

✍️ *Your answer:*

## 7. Stretch: write the flight lines yourself

**Exercise 5 (stretch):** write `my_flight_lines(area, spacing, heading)` that returns a list of `LineString`s like `mt.flight_lines`. Steps:
1. `rot = affinity.rotate(area, -heading, origin=area.centroid)`: now the flight direction is the x axis.
2. `rot.bounds` gives `x0, y0, x1, y1`. Number of lines `n = ceil((y1 - y0) / spacing)`; their y values centred between `y0` and `y1`.
3. For each y: `LineString([(x0 - 1, y), (x1 + 1, y)]).intersection(rot)`. If the result is a `MultiLineString` (the line crosses a concave part), keep the longest part.
4. Reverse every second line (`LineString(list(seg.coords)[::-1])`) and rotate each line back with `+heading` and **the same origin**.

`mt.same_lines` compares your lines with the reference.

In [ ]:
from shapely import affinity
from shapely.geometry import LineString, MultiLineString

# YOUR CODE HERE

mt.same_lines(my_flight_lines(area, spacing, heading), lines)